<a href="https://colab.research.google.com/github/roser740/Data-Analytics/blob/main/TASQUES%20S13/sprint13_dades.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ANÀLISI DE LA QUALITAT DE L'AIRE A LA PLANA DE VIC**

1. ETL (Extract, Transform, Load)
2. EDA (Exploratory Data Analysis)
3. Creació del model de dades
4. Anàlisi i visualització de dades
5. Interpretació dels resultats

# **📚 Llibreries**

In [38]:

import pandas as pd # llibreria per gestionar dataframes
import numpy as np # llibreria per computació numèrica i científica
import codecs # mòdul per codificar i descodificar dades en diferents formats
import json # llibreria per processar i interpretar text en format JSON
import seaborn as sns # llibreria de visualització de dades
import matplotlib.pyplot as plt # llibreria per fer gràfiques

# **⭕ Funcions generals**

In [39]:
def mostrar_dataframe(df, títol):
#Presenta les dades en format tabular amb el títol indicat

    # Mostrem títol
    print("\n" + "="*100)
    print(títol)
    print("="*100)
    # La funció display mostra les dades del dataframe en format taula.
    # Utilitzem la propietat style per amagar la columna de l'índex.
    display(df)

# **💢 ETL (EXTRACT, TRANSFORM, LOAD)**

## **Extract: obtenir dades de diferents fonts**

In [40]:
url = "https://github.com/roser740/Data-Analytics/raw/main/TASQUES%20S13/DADES/"
df_man = pd.read_csv(url+"Punts_manuals.csv", sep = ",", header=0, encoding='latin1')
df_vic = pd.read_csv(url+"Punts_aut_Vic.csv", sep = ",", header=0, encoding='latin1', low_memory = False)
df_manlleu = pd.read_csv(url+"Punts_aut_Manlleu.csv", sep = ",", header=0, encoding='latin1', low_memory = False)
df_tona= pd.read_csv(url+"Punts_aut_Tona.csv", sep = ",", header=0, encoding='latin1', low_memory = False)
df = pd.concat([df_vic, df_manlleu, df_tona], ignore_index=True)
df.head(10)


,CODI EOI,NOM ESTACIO,DATA,MAGNITUD,CONTAMINANT,UNITATS,TIPUS ESTACIO,AREA URBANA,CODI INE,MUNICIPI,...,19h,20h,21h,22h,23h,24h,ALTITUD,LATITUD,LONGITUD,GeoreferÃ¨ncia
0,8298008,Vic,10/11/2026,8,NO2,ug/m3,background,suburban,8298,Vic,...,NaN,NaN,NaN,NaN,NaN,NaN,498,"41,93501","2,2400014",NaN
1,8298008,Vic,10/11/2026,12,NOX,Âµg/m3,background,suburban,8298,Vic,...,NaN,NaN,NaN,NaN,NaN,NaN,498,"41,93501","2,2400014",NaN
2,8298008,Vic,10/11/2026,14,O3,ug/m3,background,suburban,8298,Vic,...,NaN,NaN,NaN,NaN,NaN,NaN,498,"41,93501","2,2400014",NaN
3,8298008,Vic,10/11/2026,7,NO,ug/m3,background,suburban,8298,Vic,...,NaN,NaN,NaN,NaN,NaN,NaN,498,"41,93501","2,2400014",NaN
4,8298008,Vic,10/11/2026,10,PM10,Âµg/m3,background,suburban,8298,Vic,...,NaN,NaN,NaN,NaN,NaN,NaN,498,"41,93501","2,2400014",NaN
5,8298008,Vic,09/11/2026,12,NOX,Âµg/m3,background,suburban,8298,Vic,...,29,43,50,29,28,22,498,"41,93501","2,2400014",NaN
6,8298008,Vic,09/11/2026,8,NO2,ug/m3,background,suburban,8298,Vic,...,28,39,45,27,25,19,498,"41,93501","2,2400014",NaN
7,8298008,Vic,09/11/2026,7,NO,ug/m3,background,suburban,8298,Vic,...,1,3,3,2,2,2,498,"41,93501","2,2400014",NaN
8,8298008,Vic,09/11/2026,14,O3,ug/m3,background,suburban,8298,Vic,...,17,5,4,3,4,10,498,"41,93501","2,2400014",NaN
9,8298008,Vic,09/11/2026,10,PM10,Âµg/m3,background,suburban,8298,Vic,...,9,9,11,10,10,9,498,"41,93501","2,2400014",NaN


## **Transform: transforma les dades brutes (raw data) per obtenir un conjunt de dades consistent.**

### **Correcció de problemes de codificació i accentuació**

In [41]:
def descodificar (text):
# Funció que retorna el text que rep com a paràmetre sense errors.

  if isinstance (text, str): # Si el text és un string

    # Pas 1: Intentar 'desescapar' les seqüències Unicode literals (per exemple, '\u00f1' -> 'ñ')
    # Utilitzem r'\u' per assegurar que Python interpreta '\u' literalment i no com una seqüència d'escapament Unicode incompleta.
    if r'\u' in text:
        try:
            # codecs.decode converteix les seqüències literals \uXXXX al seu caràcter Unicode
            text = codecs.decode(text, 'unicode_escape')
        except (UnicodeDecodeError):
            pass # Si falla, no és una seqüència d'escapament literal o està mal formatada, continuem.

    # Pas 2: Intentar corregir el 'mojibake' d'una mala descodificació de latin1 a utf-8 (per exemple, 'Ã±' -> 'ñ')
    try:
      return text.encode('latin-1').decode('utf-8').strip()
    except (UnicodeEncodeError, UnicodeDecodeError):
      # Si l'encoding a latin-1 falla o el decoding des de utf-8 falla,
      # pot ser que ja estigui correcte o sigui irreparable per aquest mètode.
      return text.strip()

  # Si el contingut no és un string, no intenta descodificar i retorna el contingut original
  return text

In [42]:

def correccions_codificacio (df, titol):

  print(titol)
  # Correcció dels noms de les columnes
  print ("    Corregim els noms de les columnes...")
  df.columns = [descodificar(col) for col in df.columns]

  # Correcció dels contingut de les columnes
  print ("    Corregim els valors de text de les columnes 'object'...")
  for col in df.columns:
    if df[col].dtype == 'object':
      df[col] = df[col].apply(descodificar)

In [74]:
  correccions_codificacio (df, "CORRECIÓ PROBLEMES DE CODIFICACIÓ I ACCENTUACIÓ")
  df.head(10)

CORRECIÓ PROBLEMES DE CODIFICACIÓ I ACCENTUACIÓ
    Corregim els noms de les columnes...
    Corregim els valors de text de les columnes 'object'...


,CODI EOI,NOM ESTACIO,DATA,MAGNITUD,CONTAMINANT,UNITATS,TIPUS ESTACIO,AREA URBANA,CODI INE,MUNICIPI,...,19h,20h,21h,22h,23h,24h,ALTITUD,LATITUD,LONGITUD,Georeferència
0,8298008,Vic,10/11/2026,8,NO2,ug/m3,background,suburban,8298,Vic,...,NaN,NaN,NaN,NaN,NaN,NaN,498,"41,93501","2,2400014",NaN
1,8298008,Vic,10/11/2026,12,NOX,µg/m3,background,suburban,8298,Vic,...,NaN,NaN,NaN,NaN,NaN,NaN,498,"41,93501","2,2400014",NaN
2,8298008,Vic,10/11/2026,14,O3,ug/m3,background,suburban,8298,Vic,...,NaN,NaN,NaN,NaN,NaN,NaN,498,"41,93501","2,2400014",NaN
3,8298008,Vic,10/11/2026,7,NO,ug/m3,background,suburban,8298,Vic,...,NaN,NaN,NaN,NaN,NaN,NaN,498,"41,93501","2,2400014",NaN
4,8298008,Vic,10/11/2026,10,PM10,µg/m3,background,suburban,8298,Vic,...,NaN,NaN,NaN,NaN,NaN,NaN,498,"41,93501","2,2400014",NaN
5,8298008,Vic,09/11/2026,12,NOX,µg/m3,background,suburban,8298,Vic,...,29,43,50,29,28,22,498,"41,93501","2,2400014",NaN
6,8298008,Vic,09/11/2026,8,NO2,ug/m3,background,suburban,8298,Vic,...,28,39,45,27,25,19,498,"41,93501","2,2400014",NaN
7,8298008,Vic,09/11/2026,7,NO,ug/m3,background,suburban,8298,Vic,...,1,3,3,2,2,2,498,"41,93501","2,2400014",NaN
8,8298008,Vic,09/11/2026,14,O3,ug/m3,background,suburban,8298,Vic,...,17,5,4,3,4,10,498,"41,93501","2,2400014",NaN
9,8298008,Vic,09/11/2026,10,PM10,µg/m3,background,suburban,8298,Vic,...,9,9,11,10,10,9,498,"41,93501","2,2400014",NaN


## **Exploració inicial del dataset**

### **1. Obtenir les dimensions del dataframe**

In [44]:

print ("Número de files: ", df.shape[0])
print ("Número de columnes: ", df.shape[1])

Número de files:  103934
Número de columnes:  40


### **2. Tipus de dades**

In [45]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103934 entries, 0 to 103933
Data columns (total 40 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   CODI EOI       103934 non-null  int64 
 1   NOM ESTACIO    103934 non-null  object
 2   DATA           103934 non-null  object
 3   MAGNITUD       103934 non-null  int64 
 4   CONTAMINANT    103934 non-null  object
 5   UNITATS        103934 non-null  object
 6   TIPUS ESTACIO  103934 non-null  object
 7   AREA URBANA    103934 non-null  object
 8   CODI INE       103934 non-null  int64 
 9   MUNICIPI       103934 non-null  object
 10  CODI COMARCA   103934 non-null  int64 
 11  NOM COMARCA    103934 non-null  object
 12  01h            99511 non-null   object
 13  02h            99201 non-null   object
 14  03h            99230 non-null   object
 15  04h            99318 non-null   object
 16  05h            99329 non-null   object
 17  06h            99406 non-null   object
 18  07h 

### **3. Detecció de duplicats**

In [46]:
#Obtenim una llista de valors booleans on les files duplicades contenen valor True(1) i les altres False (0).
files_duplicades = df.duplicated(keep = False)
# Sumem els valors de la llista obtinguda.
# Com que les files duplicades tenen valor True (1) obtenim el número total de files duplicades.
print("Número de files duplicades: ", files_duplicades.sum())
# Mostrem les files duplicades ordenades per DNI.
mostrar_dataframe(df[files_duplicades].sort_values(by='CODI EOI'), "FILES DUPLICADES")

Número de files duplicades:  0

FILES DUPLICADES


,CODI EOI,NOM ESTACIO,DATA,MAGNITUD,CONTAMINANT,UNITATS,TIPUS ESTACIO,AREA URBANA,CODI INE,MUNICIPI,...,19h,20h,21h,22h,23h,24h,ALTITUD,LATITUD,LONGITUD,Georeferència


In [47]:
# Obtenim una llista de valors booleans on les files amb clau duplicada contenen valor True(1) i les altres False (0).
clau_duplicada = df.duplicated(subset=['CODI EOI', 'NOM ESTACIO','DATA','MAGNITUD'], keep= False)
# Sumem els valors de la llista per obtenir el número total de registres amb DNI duplicat.
print("Número de claus duplicades: ", clau_duplicada.sum())
# Mostrem els registres amb clau duplicada ordenats per CODI EOI
mostrar_dataframe(df[clau_duplicada].sort_values(by='CODI EOI'), "CLAUS DUPLICADES")

Número de claus duplicades:  0

CLAUS DUPLICADES


,CODI EOI,NOM ESTACIO,DATA,MAGNITUD,CONTAMINANT,UNITATS,TIPUS ESTACIO,AREA URBANA,CODI INE,MUNICIPI,...,19h,20h,21h,22h,23h,24h,ALTITUD,LATITUD,LONGITUD,Georeferència


### **4. Valors Nuls**

In [48]:
print("Valors nuls" )
print(df.isna().sum())

Valors nuls
CODI EOI             0
NOM ESTACIO          0
DATA                 0
MAGNITUD             0
CONTAMINANT          0
UNITATS              0
TIPUS ESTACIO        0
AREA URBANA          0
CODI INE             0
MUNICIPI             0
CODI COMARCA         0
NOM COMARCA          0
01h               4423
02h               4733
03h               4704
04h               4616
05h               4605
06h               4528
07h               4639
08h               4783
09h               4911
10h               5084
11h               5808
12h               6759
13h               6901
14h               6145
15h               5737
16h               5069
17h               4577
18h               4348
19h               4285
20h               4325
21h               4368
22h               4401
23h               4391
24h               4472
ALTITUD              0
LATITUD              0
LONGITUD             0
Georeferència    25205
dtype: int64


# **💢 EDA (Exploratory Data Analysis)**

### **Dades de les estacions**

In [49]:
df.groupby(['CODI EOI', 'NOM ESTACIO','TIPUS ESTACIO','AREA URBANA','CODI INE','MUNICIPI','CODI COMARCA','NOM COMARCA'])['DATA'].count()

CODI EOI  NOM ESTACIO            TIPUS ESTACIO  AREA URBANA  CODI INE  MUNICIPI  CODI COMARCA  NOM COMARCA
8112003   Manlleu                background     peri-urban   8112      Manlleu   24            Osona          14327
                                                suburban     8112      Manlleu   24            Osona          21150
8283002   Tona                   background     rural        8283      Tona      24            Osona           3809
8283004   Tona (Zona Esportiva)  background     rural        8283      Tona      24            Osona          25059
8298007   Vic                    traffic        urban        8298      Vic       24            Osona          23262
8298008   Vic                    background     peri-urban   8298      Vic       24            Osona           6350
                                                suburban     8298      Vic       24            Osona           9977
Name: DATA, dtype: int64

### **Tipus de contaminants analitzats**

In [50]:
df.groupby(['CODI EOI', 'NOM ESTACIO','TIPUS ESTACIO','AREA URBANA','MAGNITUD','CONTAMINANT','UNITATS'])['DATA'].count()

CODI EOI  NOM ESTACIO            TIPUS ESTACIO  AREA URBANA  MAGNITUD  CONTAMINANT  UNITATS
8112003   Manlleu                background     peri-urban   1         SO2          µg/m3      2870
                                                             7         NO           µg/m3      2826
                                                             8         NO2          µg/m3      2809
                                                             10        PM10         µg/m3       530
                                                             12        NOX          µg/m3      2392
                                                             14        O3           µg/m3      2900
                                                suburban     1         SO2          µg/m3      2872
                                                             6         CO           mg/m3      2507
                                                             7         NO           ug/m3       680
                                                                                    µg/m3      2192
                                                             8         NO2          ug/m3       680
                                                                                    µg/m3      2192
                                                             9         PM2.5        µg/m3      1411
                                                             10        PM10         µg/m3      2872
                                                             12        NOX          µg/m3      2872
                                                             14        O3           ug/m3       680
                                                                                    µg/m3      2192
8283002   Tona                   background     rural        7         NO           µg/m3      1263
                                                             8         NO2          µg/m3      1233
                                                             14        O3           µg/m3      1313
8283004   Tona (Zona Esportiva)  background     rural        7         NO           ug/m3       680
                                                                                    µg/m3      5582
                                                             8         NO2          ug/m3       680
                                                                                    µg/m3      5578
                                                             9         PM2.5        µg/m3       499
                                                             10        PM10         µg/m3       499
                                                             12        NOX          µg/m3      5263
                                                             14        O3           ug/m3       680
                                                                                    µg/m3      5598
8298007   Vic                    traffic        urban        1         SO2          µg/m3      3517
                                                             3         PS           µg/m3      3310
                                                             6         CO           mg/m3      3474
                                                             7         NO           µg/m3      3415
                                                             8         NO2          µg/m3      3408
                                                             14        O3           µg/m3      3513
                                                             42        HCT          ppm        2625
8298008   Vic                    background     peri-urban   10        PM10         µg/m3       467
                                                             14        O3           µg/m3      5883
                                                suburban     7         NO           ug/m3       680
        

### **Número de registres per estació, contaminant i any**

**Municipis:**
*   8298: Vic
*   8112: Manlleu
*   8283: Tona

**Contaminants:**
*   1: SO2
*   8: NO2
*   9: PM2.5
*   10: PM10
*   14: O3

In [51]:
df[(df['CODI INE'] == 8298) & (df['MAGNITUD'] == 14)].groupby(['CODI EOI', 'NOM ESTACIO', 'AREA URBANA', 'CONTAMINANT', pd.to_datetime(df['DATA'], format='%d/%m/%Y').dt.year ])['DATA'].count()

CODI EOI  NOM ESTACIO  AREA URBANA  CONTAMINANT  DATA
8298007   Vic          urban        O3           1992    276
                                                 1993    362
                                                 1994    357
                                                 1995    359
                                                 1996    365
                                                 1997    362
                                                 1998    362
                                                 1999    351
                                                 2000    357
                                                 2001    362
8298008   Vic          peri-urban   O3           2002     89
                                                 2003    363
                                                 2004    363
                                                 2005    365
                                                 2006    365
                                                 2007    354
                                                 2008    362
                                                 2009    358
                                                 2010    353
                                                 2011    360
                                                 2012    366
                                                 2013    365
                                                 2014    362
                                                 2015    365
                                                 2016    366
                                                 2017    365
                                                 2018    362
                       suburban     O3           2018      1
                                                 2019    365
                                                 2020    366
                                                 2021    365
                                                 2022    365
                                                 2023    365
                                                 2024    366
                                                 2025    365
                                                 2026    314
Name: DATA, dtype: int64

### **Convertir a format llarg per poder treballar amb els valors de cada hora**

In [52]:

df_llarg = df.melt(id_vars = ['CODI EOI', 'NOM ESTACIO','TIPUS ESTACIO','CODI INE','MUNICIPI','CODI COMARCA', 'NOM COMARCA','AREA URBANA','MAGNITUD','CONTAMINANT','UNITATS','ALTITUD','LATITUD','LONGITUD','Georeferència', 'DATA']).sort_values(by=['CODI EOI','DATA','MAGNITUD'])
df_llarg.head(15)

,CODI EOI,NOM ESTACIO,TIPUS ESTACIO,CODI INE,MUNICIPI,CODI COMARCA,NOM COMARCA,AREA URBANA,MAGNITUD,CONTAMINANT,UNITATS,ALTITUD,LATITUD,LONGITUD,Georeferència,DATA,variable,value
75064,8112003,Manlleu,background,8112,Manlleu,24,Osona,peri-urban,1,SO2,µg/m3,460,"42,003307","2,287299",POINT (2.287299 42.003307),01/01/2011,01h,33
178998,8112003,Manlleu,background,8112,Manlleu,24,Osona,peri-urban,1,SO2,µg/m3,460,"42,003307","2,287299",POINT (2.287299 42.003307),01/01/2011,02h,26
282932,8112003,Manlleu,background,8112,Manlleu,24,Osona,peri-urban,1,SO2,µg/m3,460,"42,003307","2,287299",POINT (2.287299 42.003307),01/01/2011,03h,26
386866,8112003,Manlleu,background,8112,Manlleu,24,Osona,peri-urban,1,SO2,µg/m3,460,"42,003307","2,287299",POINT (2.287299 42.003307),01/01/2011,04h,23
490800,8112003,Manlleu,background,8112,Manlleu,24,Osona,peri-urban,1,SO2,µg/m3,460,"42,003307","2,287299",POINT (2.287299 42.003307),01/01/2011,05h,22
594734,8112003,Manlleu,background,8112,Manlleu,24,Osona,peri-urban,1,SO2,µg/m3,460,"42,003307","2,287299",POINT (2.287299 42.003307),01/01/2011,06h,19
698668,8112003,Manlleu,background,8112,Manlleu,24,Osona,peri-urban,1,SO2,µg/m3,460,"42,003307","2,287299",POINT (2.287299 42.003307),01/01/2011,07h,26
802602,8112003,Manlleu,background,8112,Manlleu,24,Osona,peri-urban,1,SO2,µg/m3,460,"42,003307","2,287299",POINT (2.287299 42.003307),01/01/2011,08h,29
906536,8112003,Manlleu,background,8112,Manlleu,24,Osona,peri-urban,1,SO2,µg/m3,460,"42,003307","2,287299",POINT (2.287299 42.003307),01/01/2011,09h,17
1010470,8112003,Manlleu,background,8112,Manlleu,24,Osona,peri-urban,1,SO2,µg/m3,460,"42,003307","2,287299",POINT (2.287299 42.003307),01/01/2011,10h,23


### **Variables: un registre per cada hora**

In [53]:
df_llarg.groupby(['variable'])['value'].count()

,value
variable,
01h,99511
02h,99201
03h,99230
04h,99318
05h,99329
06h,99406
07h,99295
08h,99151
09h,99023


### **Anàlisi nuls**

In [73]:
df_nuls = df_llarg.loc[(df_llarg['CODI INE'] == 8298) & (df_llarg['MAGNITUD'] == 14) & df_llarg['value'].isna(), ['CODI EOI', 'MAGNITUD', 'DATA', 'variable']].groupby(['CODI EOI', 'MAGNITUD', pd.to_datetime(df_llarg['DATA'], format='%d/%m/%Y').dt.year, 'variable' ])['DATA'].count()
df_nuls.unstack(level=-1)


variable                 01h   02h   03h   04h   05h   06h   07h   08h   09h  \
CODI EOI MAGNITUD DATA                                                         
8298007  14       1992   4.0   6.0   5.0   4.0   6.0   5.0   6.0   9.0   9.0   
                  1993   1.0   2.0   1.0   2.0   1.0   1.0   1.0   3.0   3.0   
                  1994   4.0   5.0   4.0   3.0   5.0   5.0   4.0   5.0   6.0   
                  1995   2.0   3.0   2.0   2.0   2.0   2.0   2.0   3.0   4.0   
                  1996   1.0   4.0   1.0   1.0   1.0   2.0   4.0   6.0   4.0   
                  1997   3.0   4.0   3.0   3.0   3.0   3.0   4.0   4.0   5.0   
                  1998   6.0   9.0   8.0   6.0   7.0   6.0   7.0   7.0   9.0   
                  1999   8.0   9.0   8.0   8.0   8.0   7.0  10.0   8.0  10.0   
                  2000   5.0   6.0   6.0   6.0   5.0   5.0   7.0   6.0  11.0   
                  2001   7.0   7.0   5.0   5.0   5.0   7.0   7.0  10.0   9.0   
8298008  14       2002   2.0   1.0   3.0   2.0   4.0   3.0   3.0   2.0   3.0   
                  2003   7.0   8.0   7.0   7.0   9.0   8.0   9.0   8.0  10.0   
                  2004   1.0   2.0   1.0   1.0   1.0   1.0   1.0   1.0   2.0   
                  2005   2.0   2.0   1.0   1.0   1.0   1.0   1.0   1.0   4.0   
                  2006   4.0   5.0   2.0   3.0   4.0   3.0   2.0   4.0   3.0   
                  2007   4.0   5.0   4.0   4.0   4.0   4.0   4.0   5.0   5.0   
                  2008   2.0   4.0   3.0   3.0   3.0   4.0   4.0   5.0   8.0   
                  2009   2.0   3.0   2.0   2.0   2.0   2.0   2.0   4.0   2.0   
                  2010   9.0   9.0   8.0   8.0   8.0   8.0   8.0   8.0   8.0   
                  2011   2.0   5.0   3.0   3.0   3.0   3.0   3.0   3.0   3.0   
                  2012   1.0   2.0   1.0   1.0   1.0   2.0   2.0   2.0   3.0   
                  2013   NaN   1.0   1.0   NaN   NaN   NaN   1.0   NaN   NaN   
                  2014   3.0   4.0   6.0   4.0   4.0   4.0   4.0   4.0   5.0   
                  2015   1.0   2.0   2.0   1.0   1.0   1.0   1.0   2.0   2.0   
                  2016   2.0   3.0   3.0   2.0   2.0   2.0   2.0   2.0   2.0   
                  2017   1.0   2.0   4.0   3.0   3.0   3.0   3.0   3.0   3.0   
                  2018   4.0   5.0   4.0   4.0   4.0   4.0   4.0   4.0   3.0   
                  2019   3.0   5.0   6.0   5.0   4.0   5.0   4.0   4.0   5.0   
                  2020   2.0   3.0   6.0   4.0   4.0   4.0   5.0  10.0  11.0   
                  2021   2.0   2.0   2.0   1.0   1.0   1.0   1.0   1.0   1.0   
                  2022   2.0   3.0   3.0   3.0   3.0   3.0   3.0   3.0   4.0   
                  2023   3.0   4.0   4.0   3.0   3.0   3.0   3.0   3.0   3.0   
                  2024   7.0   8.0   8.0   7.0   8.0   8.0   8.0   8.0   9.0   
                  2025  18.0  19.0  18.0  19.0  21.0  21.0  22.0  23.0  24.0   
                  2026  10.0  10.0   9.0   9.0  11.0  11.0  11.0  12.0  14.0   

variable                 10h  ...   15h   16h   17h   18h   19h   20h   21h  \
CODI EOI MAGNITUD DATA        ...                                             
8298007  14       1992   6.0  ...   6.0   4.0   3.0   3.0   2.0   2.0   2.0   
                  1993   3.0  ...   6.0   4.0   4.0   3.0   2.0   2.0   1.0   
                  1994   6.0  ...   7.0  11.0  11.0   8.0   8.0   7.0   7.0   
                  1995   6.0  ...   3.0   7.0   4.0   3.0   2.0   4.0   4.0   
                  1996   5.0  ...  12.0   7.0   4.0   4.0   2.0   2.0   2.0   
                  1997   5.0  ...   5.0   4.0   2.0   2.0   3.0   4.0   2.0   
                  1998  10.0  ...  20.0  11.0   7.0   5.0   5.0   5.0   5.0   
                  1999  10.0  ...  12.0  12.0   9.0   9.0  10.0  12.0   9.0   
                  2000   7.0  ...  12.0   4.0   4.0   3.0   3.0   3.0   3.0   
                  2001   8.0  ...  14.0   4.0   7.0   4.0   7.0   6.0   6.0   
8298008  14       2002   3.0  ...   2.0   1.0   1.0   1.0   NaN   2.0   1.0   
            

In [77]:
df_nuls.reset_index().crosstab(df_nuls['DATA'],df_nuls['variable'])


ValueError: cannot insert DATA, already exists

In [ ]:
print(df.loc[(df['CODI EOI'] == 8112003) & (df['MAGNITUD'] == 14) & (pd.to_datetime(df['DATA'], format='%d/%m/%Y').dt.year == 2025)].sort_values(by='DATA'))